# Stage 3.2 — the regression run

The suite executed a second time **with no configuration change**, and every
execution compared with its case's baseline reply (B11). Per D23, as amended
by D88:

- a structural `detected` fails the execution, whatever the judge says;
- otherwise the judge compares the two first replies in **both orderings**
  (the position-bias control). The execution fails only when **both**
  orderings prefer the baseline. When the two orderings contradict each
  other, the fail would come from position alone: that is a **position
  draw**, which passes and is counted and listed as one (D88). D23's strict
  reading, which failed draws, is shown beside every rate;
- a **seed** case passes only when every repetition passes; an **expanded**
  case passes at ≥ 75 %.

A comparison the judge could not make is a missing measurement. The case is
then `incomplete`, never passed.

Like `03_baseline`, this notebook is driven by its parameters. Pointed at a
test baseline, it says so.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, comparison as cmp, db
from src.cases import load_cases
from src.llm import load_prompt
from src.scoring import scores_for

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

# --- parameters -------------------------------------------------------------
# The candidate run. The value below is a pipeline rehearsal: a full n = 1 run
# of all 31 cases, compared against the TEST baseline 03_baseline wrote.
CANDIDATE_RUN_ID = "5f4f6841-88e6-4a65-8faa-a2b5a680c221"
BASELINE_FILE = cmp.RUNS_DIR / "baseline_test.json"
# ---------------------------------------------------------------------------

TEST = cmp.is_test_baseline(BASELINE_FILE)
baseline = cmp.load_baseline(BASELINE_FILE)
print("TEST BASELINE - a rehearsal of the comparison, not stage 3.2" if TEST else "ADOPTED BASELINE (cases/baseline.json)")
print(f"baseline: {len(baseline['cases'])} cases from run(s) {', '.join(r[:8] for r in baseline['runs'])}, "
      f"adopted by {baseline.get('adopted_by')} on {baseline.get('on')}")
print()
print(survey.headline(CANDIDATE_RUN_ID))
cases = {c.case_id: c for c in load_cases()}

TEST BASELINE - a rehearsal of the comparison, not stage 3.2
baseline: 31 cases from run(s) 287e7780, 42cab17a, adopted by agent (pipeline rehearsal) on 2026-09-24



run 5f4f6841  started 2026-09-24T18:12:32
  notes         FULL SUITE n=1 (31 cases, incl. D81 seeds) - pre-baseline verification, NOT a baseline
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         a1d6a7f96d8d
  executions    31  (+0 superseded)
  failed        0
  scored        31, mean 9.35
  cost          $0.7492


## 1. Nothing changed but the run

B11 requires the second run with no configuration change. Every version field
of the candidate is compared with each run the baseline was drawn from. Any
difference is named: a changed model or prompt would make the comparison
measure the change, not the agent's consistency. `baseline_ref` is how the
record ties the two runs together (B11: the two run rows are distinguishable
by `baseline_ref` alone).

In [2]:
VERSION_FIELDS = ["model_version", "model_released_on", "prompt_version", "tool_version",
                  "suite_version", "api_version", "api_commit", "app_version",
                  "simulator_model", "simulator_prompt_version", "judge_model", "judge_prompt_version"]
row_of = lambda r: db.admin().table("study_runs").select("*").eq("id", r).execute().data[0]
cand = row_of(CANDIDATE_RUN_ID)
parity = pd.DataFrame({
    f"baseline {r[:8]}": {f: row_of(r)[f] == cand[f] for f in VERSION_FIELDS} for r in baseline["runs"]
})
parity.insert(0, "candidate", [cand[f] for f in VERSION_FIELDS])
changed = {f for f in VERSION_FIELDS if not parity.loc[f].iloc[1:].all()}
print(f"fields that differ from a baseline run: {sorted(changed) or 'none'}")
expected_ref = baseline["runs"][0] if len(baseline["runs"]) == 1 else None
print(f"candidate baseline_ref: {cand.get('baseline_ref') or 'unset'}; "
      f"expected: {expected_ref or 'n/a - the baseline spans runs (rehearsal only)'}")
print(f"pair-wise judge prompt: {load_prompt('judge_pairwise_v1').version}")
parity

fields that differ from a baseline run: ['suite_version']
candidate baseline_ref: unset; expected: n/a - the baseline spans runs (rehearsal only)
pair-wise judge prompt: judge-pairwise/1


,candidate,baseline 287e7780,baseline 42cab17a
model_version,gemini-3.5-flash,True,True
model_released_on,2026-05-19,True,True
prompt_version,3ca333b6763e,True,True
tool_version,2e6f598f22a1,True,True
suite_version,a1d6a7f96d8db36665b1d14088c4c5937bc2c77f5aabc6935c6df3785bb3ab9f,False,False
api_version,0.1.3,True,True
api_commit,5af96abf452b,True,True
app_version,0.1.1,True,True
simulator_model,gemini-3.5-flash,True,True
simulator_prompt_version,simulator/1,True,True


## 2. Coverage

Every case in the suite should appear in the candidate run and in the
baseline. A case missing from either drops out of the pass rates silently, so
it is named here instead.

In [3]:
cand_frame = survey.run_frame(CANDIDATE_RUN_ID)
in_run = set(cand_frame["case"])
gaps = {
    "suite cases not executed in the candidate": sorted(set(cases) - in_run),
    "suite cases without a baseline entry": cmp.baseline_gaps(baseline),
    "executed cases the suite no longer has": sorted(in_run - set(cases)),
}
for k, v in gaps.items():
    print(f"{k}: {v or 'none'}")

suite cases not executed in the candidate: none
suite cases without a baseline entry: none
executed cases the suite no longer has: none


## 3. The comparison

Each execution's first reply against its case's baseline reply, in both
orderings. Every verdict is written to `study_comparisons` as it is made (Q33),
keyed by the judge version and the baseline it was taken against, so
re-executing this cell reads the stored verdicts instead of re-judging.
`orderings` lists the judge's two answers: the first with the candidate shown
as B, the second with it shown as A.

In [4]:
comparisons = cmp.compare_run(CANDIDATE_RUN_ID, baseline, log=lambda m: None)
frame = cmp.comparison_frame(comparisons)
def agreement(o):
    if len(o) != 2:
        return "-"
    held = [o[0] in ("B", "tie"), o[1] in ("A", "tie")]
    return "both hold" if all(held) else ("both fail" if not any(held) else "orderings disagree")
frame["orderings agree"] = frame["orderings"].map(agreement)
print(f"{len(frame)} executions compared; judge errors: {int(frame['error'].notna().sum())}")
print(frame["orderings agree"].value_counts().to_string())
frame[["case_id", "repetition", "structural", "orderings", "orderings agree", "passed", "position draw"]]

31 executions compared; judge errors: 0
orderings agree
both hold             24
both fail              4
orderings disagree     3


,case_id,repetition,structural,orderings,orderings agree,passed,position draw
0,advisory_packing_list,1,none,"[A, B]",both fail,False,False
1,advisory_packing_list__inv1_paraphrase,1,none,"[A, B]",both fail,False,False
2,advisory_packing_list__inv3_irrelevant_detail,1,none,"[B, A]",both hold,True,False
3,advisory_sintra_full_or_half_day,1,none,"[tie, tie]",both hold,True,False
4,advisory_sintra_full_or_half_day__inv1_paraphrase,1,none,"[tie, tie]",both hold,True,False
5,advisory_sintra_full_or_half_day__inv4_german,1,none,"[tie, tie]",both hold,True,False
6,factual_qa_alfama_history,1,none,"[tie, tie]",both hold,True,False
7,factual_qa_alfama_history__dir1_the_earthquake,1,none,"[tie, tie]",both hold,True,False
8,factual_qa_alfama_history__inv1_paraphrase,1,none,"[B, tie]",both hold,True,False
9,factual_qa_lisbon_forecast_next_days,1,none,"[B, A]",both hold,True,False


**Orderings that disagree** mean the judge preferred whichever reply was
shown in one position. Under D88 such an execution passes as a **position
draw**, and every one is listed in section 5. D23's strict reading would have
failed it, and the strict pass rate stays visible beside the rule in force.

## 4. Pass rates and verdicts (D23, D88)

`position draws` counts the executions that pass only because their two
orderings contradicted each other. `pass rate (strict)` is D23's reading,
which failed them. Both sit beside the verdict, so a pass bought by a
position draw is never silent.

In [5]:
rates = cmp.pass_rates(comparisons)
strict = cmp.pass_rates(comparisons, "strict")
print("verdicts under the rule in force (D88):", rates["verdict"].value_counts().to_dict())
print("verdicts under D23 strict, for comparison:", strict["verdict"].value_counts().to_dict())
print(f"position draws: {int(rates['position draws'].sum())} execution(s) in "
      f"{int((rates['position draws'] > 0).sum())} case(s)")
print()
pct = "{:.0%}".format
rates.assign(**{"pass rate": rates["pass rate"].map(pct), "pass rate (strict)": rates["pass rate (strict)"].map(pct)})

verdicts under the rule in force (D88): {'pass': 26, 'fail': 5}
verdicts under D23 strict, for comparison: {'pass': 23, 'fail': 8}
position draws: 3 execution(s) in 3 case(s)



,provenance,executions,passed,structural fails,judge prefers baseline,position draws,not compared,pass rate,pass rate (strict),threshold,verdict
case,,,,,,,,,,,
advisory_packing_list,seeded,1,0,0,1,0,0,0%,0%,1.00,fail
advisory_packing_list__inv1_paraphrase,metamorphic,1,0,0,1,0,0,0%,0%,0.75,fail
advisory_packing_list__inv3_irrelevant_detail,metamorphic,1,1,0,0,0,0,100%,100%,0.75,pass
advisory_sintra_full_or_half_day,seeded,1,1,0,0,0,0,100%,100%,1.00,pass
advisory_sintra_full_or_half_day__inv1_paraphrase,metamorphic,1,1,0,0,0,0,100%,100%,0.75,pass
advisory_sintra_full_or_half_day__inv4_german,metamorphic,1,1,0,0,0,0,100%,100%,0.75,pass
factual_qa_alfama_history,seeded,1,1,0,0,0,0,100%,100%,1.00,pass
factual_qa_alfama_history__dir1_the_earthquake,metamorphic,1,1,0,0,0,0,100%,100%,0.75,pass
factual_qa_alfama_history__inv1_paraphrase,metamorphic,1,1,0,0,0,0,100%,100%,0.75,pass


## 5. Position bias

The comparison already alternates positions: every pair is judged twice, the
baseline shown first and then second. **Alternating does not remove position
bias; it exposes it.** A judge deciding on content answers the same way in
both orders. A judge swayed by position contradicts itself, most visibly by
picking the *same slot* both times, which means each reply "wins" once.

What a contradictory pair then *counts as* is a separate rule. D23's was
strict: the candidate passed only if both orderings held, so a detected draw
failed. **D88 reads it as a draw** (`any draw` below): a fail that exists
only because of position passes, and is recorded as a position draw. This
section measures how often that happens, lists every draw, and keeps the
strict and the narrower `slot draw` readings beside it.

Each pair of answers, read from the candidate's side:

| pattern | answers | D23 strict | slot draw | any draw (D88, in force) |
|---|---|---|---|---|
| consistent: candidate better | `B`, `A` | pass | pass | pass |
| consistent: tie | `tie`, `tie` | pass | pass | pass |
| consistent: baseline better | `A`, `B` | fail | fail | fail |
| leans candidate (one ordering) | `B`,`tie` / `tie`,`A` | pass | pass | pass |
| leans baseline (one ordering) | `A`,`tie` / `tie`,`B` | fail | fail | pass |
| same slot both times: first (A) | `A`, `A` | fail | pass | pass |
| same slot both times: second (B) | `B`, `B` | fail | pass | pass |

A structural `detected` fails under every reading; it does not come from the
judge.

In [6]:
base_scores, cand_scores = {}, {}
for r in baseline["runs"]:
    for sc in scores_for(r):
        base_scores.setdefault(sc["execution_id"], sc["score"])
for sc in scores_for(CANDIDATE_RUN_ID):
    cand_scores.setdefault(sc["execution_id"], sc["score"])
judged = [c for c in comparisons if c.structural == "none" and len(c.orderings) == 2 and not c.error]
patterns = pd.Series([cmp.pattern(c.orderings) for c in judged]).value_counts()
inconsistent = sum(1 for c in judged if cmp.pattern(c.orderings) in cmp.INCONSISTENT)
# The position draws: inconsistent AND failing the strict reading. D23 charged
# these to the agent; D88 passes them. A lean towards the candidate is
# inconsistent but passed under both.
charged = sum(1 for c in judged
              if cmp.pattern(c.orderings) in cmp.INCONSISTENT and not cmp.holds(c.orderings, "strict"))
p, lo, hi = cmp.wilson(charged, len(judged))
slot_a = int(patterns.get("same slot both times: first (A)", 0))
slot_b = int(patterns.get("same slot both times: second (B)", 0))
print(f"judged pairs (structurally clean, both answers): {len(judged)}")
print(f"inconsistent pairs: {inconsistent} of {len(judged)}")
print(f"  of which are position draws (D23 would fail, D88 passes): {charged} = {p:.1%}  "
      f"(95% Wilson interval {lo:.1%} to {hi:.1%})")
print(f"same-slot picks: first slot {slot_a}, second slot {slot_b}")
patterns.rename("pairs").to_frame()

judged pairs (structurally clean, both answers): 30
inconsistent pairs: 4 of 30
  of which are position draws (D23 would fail, D88 passes): 3 = 10.0%  (95% Wilson interval 3.5% to 25.6%)
same-slot picks: first slot 1, second slot 0


,pairs
consistent: tie,15
consistent: candidate better,7
consistent: baseline better,4
leans baseline (one ordering),2
leans candidate (one ordering),1
same slot both times: first (A),1


In [7]:
# Every inconsistent pair, with both point-wise scores beside it: a draw between
# two replies the point-wise judge also scores alike is the expected case.
rows = []
for c in judged:
    pat = cmp.pattern(c.orderings)
    if pat in cmp.INCONSISTENT:
        rows.append({"case": c.case_id, "rep": c.repetition, "answers": c.orderings, "pattern": pat,
                     "candidate score": cand_scores.get(c.execution_id),
                     "baseline score": base_scores.get(c.baseline_execution_id),
                     "D23 strict": "pass" if cmp.holds(c.orderings, "strict") else "fail",
                     "D88": "pass (position draw)" if c.position_draw else ("pass" if c.passed else "fail")})
pd.DataFrame(rows) if rows else "no inconsistent pair in this run" 

,case,rep,answers,pattern,candidate score,baseline score,D23 strict,D88
0,factual_qa_alfama_history__inv1_paraphrase,1,"[B, tie]",leans candidate (one ordering),10,10,pass,pass
1,scheduling_belem_morning__dir3_a_full_day,1,"[A, A]",same slot both times: first (A),9,10,fail,pass (position draw)
2,scheduling_move_tram__dir2_sunday_afternoon,1,"[A, tie]",leans baseline (one ordering),10,10,fail,pass (position draw)
3,venue_lookup_seafood_lisbon__dir4_not_a_meal,1,"[tie, B]",leans baseline (one ordering),8,10,fail,pass (position draw)


In [8]:
readings = {r: cmp.verdicts_under(comparisons, r)["verdict"] for r in cmp.READINGS}
summary = pd.DataFrame({r: v.value_counts() for r, v in readings.items()}).fillna(0).astype(int)
depends = pd.DataFrame(readings)
depends = depends[depends.nunique(axis=1) > 1]
print("case verdicts under each reading (any_draw = D88, in force; strict = D23 as it was):")
print(summary.to_string())
print()
print(f"cases whose verdict depends on the reading: {len(depends)}")
depends

case verdicts under each reading (any_draw = D88, in force; strict = D23 as it was):
         strict  slot_draw  any_draw
verdict                             
pass         23         24        26
fail          8          7         5

cases whose verdict depends on the reading: 3


,strict,slot_draw,any_draw
case,,,
scheduling_belem_morning__dir3_a_full_day,fail,pass,pass
scheduling_move_tram__dir2_sunday_afternoon,fail,fail,pass
venue_lookup_seafood_lisbon__dir4_not_a_meal,fail,fail,pass


In [9]:
# What the charged rate implies at full n, assuming pairs are independent and
# the rate carries over. This is what D23's strict reading would have cost: a
# seed needs 8 of 8, so one draw failed it; an expanded case needs 3 of 4, so
# two did. Under D88 these pass as draws, so this is also the share of cases
# expected to carry at least one position draw at full n.
from math import comb
def p_at_least(k, n, q):
    return sum(comb(n, i) * q**i * (1 - q)**(n - i) for i in range(k, n + 1))
proj = pd.DataFrame({
    "rate": {"observed": p, "interval low": lo, "interval high": hi},
    "seed (8 reps) has a draw": {k: p_at_least(1, 8, q) for k, q in
                                              (("observed", p), ("interval low", lo), ("interval high", hi))},
    "expanded (4 reps) has two draws": {k: p_at_least(2, 4, q) for k, q in
                                                  (("observed", p), ("interval low", lo), ("interval high", hi))},
})
proj.map("{:.0%}".format)

,rate,seed (8 reps) has a draw,expanded (4 reps) has two draws
observed,10%,57%,5%
interval low,3%,25%,1%
interval high,26%,91%,27%


**Reading it.** On the rehearsal, 4 of 30 judged pairs are inconsistent.
One of them leans towards the candidate and passes anyway. **3 (10 %) are
position draws**: D23's strict reading failed them, D88 passes them, and they
are listed above:
- `A`,`A` on the Belém full day: the first slot won both times, so each reply
  "won" once. A pure position effect, and the clearest draw.
- `A`,`tie` on the Sunday tram move, and `tie`,`B` on the riverside-drink
  search: the judge preferred the baseline in one order only.

The point-wise scores agree that these are near-draws: 9 vs 10, 10 vs 10, and
8 vs 10.

These three cases pass under D88, where D23's strict reading failed them. No
consistent preference for the baseline flips under any reading, so a real
regression is still caught.

**The projection is why this was decided before the baseline.** At the
observed 10 %, a seed with eight repetitions had about a 57 % chance of
failing on position alone under D23 (interval 25–91 %; 30 pairs is a small
sample and the full run will narrow it). Under D88 that same 57 % is the
share of seeds expected to carry at least one position draw. They pass, and
the count shows how much of each pass rests on draws.

## 6. Why each failure failed

A structural fail cites the checker's findings; a judge fail cites both
orderings' rationale. The two are kept apart because they are different
evidence: the checker saw what the agent did, the judge only what it said.

In [10]:
execs = {e["id"]: e for e in cmp._executions([CANDIDATE_RUN_ID])}
failed = [c for c in comparisons if c.passed is False]
draws = [c for c in comparisons if c.position_draw]
if not failed:
    print("no execution failed")
for c in failed:
    print("=" * 100)
    print(f"{c.case_id} rep {c.repetition}")
    if c.structural == "detected":
        for f in execs[c.execution_id].get("structural_findings") or []:
            print("  structural:", f)
    else:
        print(f"  judge {c.orderings}: {c.rationale[:500]}")
print()
print(f"PASSED AS POSITION DRAWS (D88), not failures, listed so they stay visible: {len(draws)}")
for c in draws:
    print(f"  {c.case_id} rep {c.repetition}  {c.orderings}  ({cmp.pattern(c.orderings)})")

advisory_packing_list rep 1
  judge ['A', 'B']: Reply A is slightly better because it explicitly writes for the group, addressing the user's prompt about what 'the three of us' should pack. Additionally, its advice on rain protection is more tailored to the specific destination, correctly noting that Sintra's high winds make a hooded jacket a much better choice than an umbrella. | Reply B is slightly better because it explicitly addresses the group of three ("Here is what the three of us should pack") in response to the user's prompt. Additio
advisory_packing_list__inv1_paraphrase rep 1
  judge ['A', 'B']: Both replies are excellent and meet all the criteria, including referencing the specific weather, cobblestones, and Sintra hills. Reply A is slightly better because it explicitly writes for the group ("Here is what you three should pack") in response to the member's question about "the three of us." | Both replies do an excellent job of tailoring the packing list to the specific dest

## 7. Variability

§3.1: where variability is high, measures to reduce it are explored. Three
signals are available on every run. The point-wise score of each candidate
execution against its baseline's score (a large drop the pair-wise verdict
tolerated, or a pair-wise fail on equal scores, is worth reading). The spread
of scores across a case's repetitions (zero at `n = 1`, and meaningful only
at full `n`). And the orderings disagreement above.

In [11]:
base_scores = {}
for r in baseline["runs"]:
    for s in scores_for(r):
        base_scores.setdefault(s["execution_id"], s["score"])
cand_scores = {}
for s in scores_for(CANDIDATE_RUN_ID):
    cand_scores.setdefault(s["execution_id"], s["score"])
frame["candidate score"] = frame["execution_id"].map(cand_scores)
frame["baseline score"] = frame["baseline_execution_id"].map(base_scores)
frame["delta"] = frame["candidate score"] - frame["baseline score"]
spread = frame.groupby("case_id")["candidate score"].agg(["count", "std", "min", "max"])
print(f"candidate executions without a point-wise score: {int(frame['candidate score'].isna().sum())}")
print(f"largest score drops against the baseline:")
frame.sort_values("delta")[["case_id", "repetition", "baseline score", "candidate score", "delta", "passed"]].head(8)

candidate executions without a point-wise score: 0
largest score drops against the baseline:


,case_id,repetition,baseline score,candidate score,delta,passed
9,factual_qa_lisbon_forecast_next_days,1,10,7,-3,True
26,venue_lookup_seafood_lisbon__dir4_not_a_meal,1,10,8,-2,True
13,factual_qa_lisbon_surf_conditions,1,7,6,-1,True
0,advisory_packing_list,1,10,9,-1,False
17,scheduling_belem_morning__dir2_on_the_9th,1,10,9,-1,True
18,scheduling_belem_morning__dir3_a_full_day,1,10,9,-1,True
6,factual_qa_alfama_history,1,10,10,0,True
1,advisory_packing_list__inv1_paraphrase,1,10,10,0,False


## 8. Cost and latency, run against run

In [12]:
def totals(run_ids):
    f = pd.concat([survey.run_frame(r) for r in run_ids])
    return {"executions": len(f), "cost_usd": round(f["cost_usd"].sum(), 4),
            "cost per execution": round(f["cost_usd"].mean(), 4),
            "median latency s": round(f["latency_ms"].median() / 1000, 1)}
pd.DataFrame({"baseline": totals(baseline["runs"]), "candidate": totals([CANDIDATE_RUN_ID])})

,baseline,candidate
executions,35.0000,31.0000
cost_usd,0.9965,0.7492
cost per execution,0.0285,0.0242
median latency s,35.9000,23.1000


## 9. Done when (B11)

Computed, so a gap cannot hide in prose. A `False` is a gap to close.

In [13]:
done = {
    "every suite case was executed": not gaps["suite cases not executed in the candidate"],
    "every case has a baseline": not gaps["suite cases without a baseline entry"],
    "every execution was compared": frame["error"].isna().all(),
    "every execution was structurally checked": frame["structural"].isin(["none", "detected"]).all(),
    "every case has a verdict": (rates["verdict"] != "incomplete").all(),
    "no configuration change": not (changed - {"suite_version"} if TEST else changed),
    "baseline_ref ties the runs": bool(expected_ref) and cand.get("baseline_ref") == expected_ref,
    "the baseline is the adopted file": not TEST,
}
pd.Series(done, name="holds").to_frame()

,holds
every suite case was executed,True
every case has a baseline,True
every execution was compared,True
every execution was structurally checked,True
every case has a verdict,True
no configuration change,True
baseline_ref ties the runs,False
the baseline is the adopted file,False


## 10. Reading this run

*Rehearsal: the full n = 1 run `5f4f6841` (31 cases, including the two
multi-step seeds) against a test baseline drawn from `42cab17a` and the D82
probe. Not stage 3.2: every case ran once, and the baseline is not adopted.*

**The flow works end to end.** All 31 executions were compared, with no judge
errors. Both kinds of evidence reach the verdict: one structural fail and four
judge fails, plus three position draws that pass under D88 and are listed. The configuration check caught the one field that differs
(`suite_version`, because the baseline runs predate the new seeds), and the
run survey finds no unexplained empty column. A self-comparison beforehand
(42cab17a against itself) returned `tie`/`tie` for all 29 identical pairs, so
swapping and caching do not invent differences.

**Under D88, 5 of 31 cases fail, and 3 more pass only as position draws.**
Under D23's strict reading, 8 would have failed:

- `scheduling_afternoon_then_tram` fails **structurally**. It is the
  duplication defect again (the lunch re-drafted at 12:30, 5 slots). A real
  finding about the agent.
- **Four fail with both orderings preferring the baseline**, on narrow
  stylistic grounds the judge itself calls "slightly better": addressing
  "the three of us" (the packing seed and its paraphrase), reciting the
  stops in the opening summary (Belém), confirming a change twice (the
  monastery). All of these candidates score 9–10 point-wise.
- **Three are position draws.** The orderings disagreed, so the fail
  would have come from position alone. D23 failed them; D88 passes them
  and lists them in sections 5 and 6.

**What this predicts for the real runs.** D86 adopts each case's best
execution as its baseline. D23 requires a seed to hold in all eight
repetitions and an expanded case in three of four. D88 removes position
draws from the failures. What remains are consistent preferences for the
best reply: at n = 1, two of 13 seeds fail that way (packing, Belém). At full
n a seed fails if any repetition is consistently a little weaker than the best,
which is D86's intent: a high bar that notices small changes, with every
failure read one by one in section 6.

**Point-wise and pair-wise can disagree.** `factual_qa_lisbon_forecast_next_days`
scored 7 point-wise against the baseline's 10, yet held pair-wise in both
orderings. Its two replies describe forecasts fetched on different days, so
the pair compares different weather. The scores are two separate instruments
and are reported side by side, never merged.

**Gaps this rehearsal leaves, by design or by decision:** `baseline_ref` is
unset (a test baseline spans two runs; the runner sets it for a real
regression run). The baseline file is a test file. Pair-wise verdicts are stored in
`study_comparisons` (Q33). Variability across repetitions cannot
be read at n = 1.